# 🛒 Shopper Spectrum: Customer Segmentation & Product Recommendation
**Domain:** E-Commerce and Retail Analytics  
**Tools:** Python, Pandas, Scikit-learn, Plotly, Streamlit  
**Techniques:** RFM Analysis, KMeans Clustering, Collaborative Filtering (Cosine Similarity)


## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv("clean_data.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
print(f"Shape: {df.shape}")
df.head()

## Step 3: Data Cleaning

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print(f"
Duplicates: {df.duplicated().sum()}")
print(f"
Cancelled orders (starting with C): {df[df.InvoiceNo.astype(str).str.startswith(chr(67))].shape[0]}")
print(f"
Data after cleaning: {df.shape}")

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Top 10 products
top_prod = df.groupby("Description")["Quantity"].sum().nlargest(10)
top_prod.plot(kind="barh", figsize=(10,6), color="purple")
plt.title("Top 10 Products by Quantity Sold")
plt.xlabel("Quantity")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly revenue trend
monthly = df.resample("ME", on="InvoiceDate")["TotalAmount"].sum()
monthly.plot(figsize=(12,4), color="indigo", linewidth=2)
plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue (GBP)")
plt.tight_layout()
plt.show()

In [ ]:
# Country-wise revenue
country_rev = df.groupby("Country")["TotalAmount"].sum().nlargest(10)
country_rev.plot(kind="bar", figsize=(10,5), color="mediumslateblue")
plt.title("Top 10 Countries by Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 5: RFM Feature Engineering

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalAmount", "sum")
).reset_index()

rfm["Monetary"] = rfm["Monetary"].round(2)
print(f"RFM shape: {rfm.shape}")
rfm.describe()

## Step 6: RFM Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rfm["Recency"].hist(ax=axes[0], bins=40, color="steelblue")
axes[0].set_title("Recency Distribution")
rfm["Frequency"].hist(ax=axes[1], bins=30, color="mediumorchid")
axes[1].set_title("Frequency Distribution")
rfm["Monetary"].hist(ax=axes[2], bins=40, color="coral")
axes[2].set_title("Monetary Distribution")
plt.tight_layout()
plt.show()

## Step 7: KMeans Clustering

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(rfm[["Recency","Frequency","Monetary"]])

# Elbow Method
inertias = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(K_range, inertias, marker="o", color="purple")
ax[0].axvline(4, color="red", linestyle="--", label="k=4 chosen")
ax[0].set_title("Elbow Curve")
ax[0].set_xlabel("Number of Clusters (k)")
ax[0].set_ylabel("Inertia")
ax[0].legend()
ax[1].plot(K_range, sil_scores, marker="o", color="green")
ax[1].set_title("Silhouette Scores")
ax[1].set_xlabel("Number of Clusters (k)")
ax[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()
print(f"Best silhouette score at k=4: {sil_scores[2]:.4f}")

In [ ]:
# Fit final model with k=4
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = km_final.fit_predict(X)

# Label clusters
from sklearn.preprocessing import MinMaxScaler
means = rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean()
ms = MinMaxScaler()
norm = ms.fit_transform(means)
scores = -norm[:,0] + norm[:,1] + norm[:,2]
ranking = scores.argsort()[::-1]
seg_names = ["High-Value","Regular","Occasional","At-Risk"]
labels_map = {int(ranking[i]): seg_names[i] for i in range(4)}

rfm["Segment"] = rfm["Cluster"].map(labels_map)
print(rfm["Segment"].value_counts())
rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].mean().round(2)

In [ ]:
# 3D scatter plot
fig = px.scatter_3d(rfm, x="Recency", y="Frequency", z="Monetary",
                    color="Segment", opacity=0.7, title="Customer Clusters (3D RFM)")
fig.show()

## Step 8: Product Recommendation (Collaborative Filtering)

In [ ]:
# Build customer-product pivot matrix
pivot = df.pivot_table(index="CustomerID", columns="Description", values="Quantity", fill_value=0)
print(f"Pivot shape: {pivot.shape}")

# Cosine similarity between products
item_sim = cosine_similarity(pivot.T)
item_sim_df = pd.DataFrame(item_sim, index=pivot.columns, columns=pivot.columns)
print("Similarity matrix shape:", item_sim_df.shape)

In [ ]:
def get_recommendations(product_name, n=5):
    if product_name not in item_sim_df.index:
        return ["Product not found"]
    scores = item_sim_df[product_name].drop(product_name).sort_values(ascending=False)
    return scores.head(n).reset_index().rename(columns={"Description":"Product", product_name:"Similarity"})

# Test
print(get_recommendations("PARTY BUNTING"))

In [ ]:
# Heatmap for top 10 products
top10 = df.groupby("Description")["TotalAmount"].sum().nlargest(10).index.tolist()
heat_data = item_sim_df.loc[top10, top10]

plt.figure(figsize=(12,8))
sns.heatmap(heat_data, annot=True, fmt=".2f", cmap="Purples",
            xticklabels=[x[:20]+"..." if len(x)>20 else x for x in top10],
            yticklabels=[x[:20]+"..." if len(x)>20 else x for x in top10])
plt.title("Product Similarity Heatmap (Top 10 Products)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 9: Save Models

In [ ]:
import pickle, json

with open("kmeans_model.pkl","wb") as f: pickle.dump(km_final, f)
with open("scaler.pkl","wb") as f: pickle.dump(scaler, f)
with open("item_sim.pkl","wb") as f: pickle.dump(item_sim_df, f)
with open("labels_map.json","w") as f: json.dump({str(k):v for k,v in labels_map.items()}, f)

rfm.to_csv("rfm_data.csv", index=False)
print("✅ All models and data saved successfully!")

## Step 10: Model Evaluation

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

sil  = silhouette_score(X, rfm["Cluster"])
db   = davies_bouldin_score(X, rfm["Cluster"])
ch   = calinski_harabasz_score(X, rfm["Cluster"])

print("=" * 40)
print("  Clustering Evaluation Metrics")
print("=" * 40)
print(f"  Silhouette Score      : {sil:.4f}  (higher = better, max 1)")
print(f"  Davies-Bouldin Score  : {db:.4f}  (lower = better)")
print(f"  Calinski-Harabasz     : {ch:.2f}  (higher = better)")
print("=" * 40)